## **GET METADATA FROM DATAHUB AND CONVERT INTO JSONL**

In [ ]:
# 01_export_columns.py
import os
import json
import requests
import re
from typing import Dict, List, Any, Optional
from dotenv import load_dotenv, find_dotenv
from datetime import datetime, timezone, timedelta  # <-- for time formatting

load_dotenv(find_dotenv(), override=True)

# ===== Config =====
GMS_BASE = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
GRAPHQL_URL = os.getenv("DATAHUB_GRAPHQL_URL", f"{GMS_BASE}/api/graphql")
TOKEN = os.getenv("DATAHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("DATAHUB_TOKEN is not set.")
PAGE_SIZE = int(os.getenv("DATAHUB_PAGE_SIZE", "100"))
TIMEOUT = 60

# Output directory for per-database JSONL files
OUTPUT_DIR = os.getenv("COLUMNS_OUTPUT_DIR", "./exports")

# Platforms using database.schema.table
RELATIONAL_3LEVEL = {"postgres", "snowflake", "redshift", "bigquery", "mssql", "mysql"}

# ===== GraphQL =====
SCROLL_QUERY = """
query scrollPG($types: [EntityType!], $query: String!, $count: Int!, $scrollId: String) {
  scrollAcrossEntities(input: { types: $types, query: $query, count: $count, scrollId: $scrollId }) {
    nextScrollId
    searchResults {
      entity {
        urn
        type
        ... on Dataset { urn }
      }
    }
  }
}
""".strip()

DATASET_META_QUERY = """
query datasetMeta($urn: String!) {
  dataset(urn: $urn) {
    urn
    platform { name properties { datasetNameDelimiter } }
    parentContainers {
      containers {
        properties { name description }
        editableProperties { description }   # pick up UI/GraphQL-edited docs
      }
    }
    schemaMetadata(version: 0) {
      fields { fieldPath description nativeDataType type }
    }
  }
}
""".strip()

# Best-effort: some versions expose dataset.lastIngested; if not, we return None.
DATASET_LAST_INGESTED_QUERY = """
query datasetSys($urn: String!) {
  dataset(urn: $urn) { lastIngested }
}
""".strip()

def _graphql(query: str, variables: dict) -> dict:
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"}
    resp = requests.post(GRAPHQL_URL, json={"query": query, "variables": variables},
                         headers=headers, timeout=TIMEOUT)
    resp.raise_for_status()
    data = resp.json()
    if data.get("errors"):
        raise RuntimeError(data["errors"][0])
    return data["data"]

def iter_dataset_urns():
    scroll_id = None
    while True:
        data = _graphql(
            SCROLL_QUERY,
            {"types": ["DATASET"], "query": "*", "count": PAGE_SIZE, "scrollId": scroll_id},
        )["scrollAcrossEntities"]
        for item in data.get("searchResults") or []:
            ent = item.get("entity") or {}
            urn = ent.get("urn")
            if urn:
                yield urn
        scroll_id = data.get("nextScrollId")
        if not scroll_id:
            break

def _parse_urn_name(urn: str) -> Optional[str]:
    # urn:li:dataset:(platformUrn,name,origin) -> name
    try:
        inner = urn[urn.index("(")+1 : urn.rindex(")")]
        _, name_part, _ = inner.split(",", 2)
        return name_part
    except Exception:
        return None

def _split_from_urn(urn: str, platform_name: Optional[str], delim: str = "."):
    # Returns (full_name, database, schema, table) from URN
    name_part = _parse_urn_name(urn)
    if not name_part:
        return None, None, None, None
    plat = (platform_name or "").lower()
    if plat in RELATIONAL_3LEVEL:
        parts = name_part.split(delim)
        if len(parts) >= 3:
            return name_part, parts[0], parts[1], delim.join(parts[2:])
        elif len(parts) == 2:
            return name_part, parts[0], None, parts[1]
        else:
            return name_part, None, None, parts[0]
    return name_part, None, None, name_part

def _best_effort_last_ingested(urn: str) -> Optional[int]:
    """Return dataset.lastIngested (ms epoch) if GraphQL supports it; else None."""
    try:
        d = _graphql(DATASET_LAST_INGESTED_QUERY, {"urn": urn})
        ds = d.get("dataset")
        if ds is not None:
            return ds.get("lastIngested")
    except Exception:
        pass
    return None

# ---- time formatting helpers (Jakarta local without timezone suffix) ----
_JAKARTA = timezone(timedelta(hours=7))
def _ms_to_local_naive(ms: Optional[int]) -> Optional[str]:
    """
    Convert epoch ms to Asia/Jakarta local time and return ISO-like string WITHOUT timezone.
    Example: 2025-09-22T15:00:00
    """
    if ms is None:
        return None
    dt_local = datetime.fromtimestamp(ms / 1000, tz=timezone.utc).astimezone(_JAKARTA)
    return dt_local.strftime("%Y-%m-%dT%H:%M:%S")

def fetch_dataset_columns(urn: str) -> List[Dict[str, Any]]:
    """
    Returns a list of column dicts:
      {database, table, column, datatype, description, database_description, last_ingested}
      (and includes {schema} ONLY for platforms where it makes sense and is present)
    """
    d = _graphql(DATASET_META_QUERY, {"urn": urn}).get("dataset")
    if not d:
        return []

    platform = d.get("platform") or {}
    platform_name = (platform.get("name") or "").lower()
    delim = (platform.get("properties") or {}).get("datasetNameDelimiter") or "."

    _, db, schema, table = _split_from_urn(urn, platform_name, delim)

    # Find database description from parent containers
    db_desc = ""
    containers = ((d.get("parentContainers") or {}).get("containers") or [])
    for c in containers:
        props = c.get("properties") or {}
        edit  = c.get("editableProperties") or {}
        if (props.get("name") or "").lower() == (db or "").lower():
            db_desc = (edit.get("description") or props.get("description") or "").strip()
            break

    # Optional fallback: first non-empty description up the chain
    if not db_desc:
        for c in containers:
            props = c.get("properties") or {}
            edit  = c.get("editableProperties") or {}
            cand = (edit.get("description") or props.get("description") or "").strip()
            if cand:
                db_desc = cand
                break

    # System recency: convert directly to Asia/Jakarta local string without tz suffix
    last_ingested_ms = _best_effort_last_ingested(urn)
    last_ingested_str = _ms_to_local_naive(last_ingested_ms)  # e.g., "2025-09-22T15:00:00"

    # MySQL: schema concept == database; omit "schema" from the output entirely
    omit_schema = (platform_name == "mysql")

    rows = []
    sm = d.get("schemaMetadata") or {}
    for f in (sm.get("fields") or []):
        row = {
            "database": db,
            "table": table,
            "column": f.get("fieldPath"),
            "datatype": f.get("nativeDataType") or (f.get("type") or ""),
            "description": f.get("description") or "",
            "database_description": db_desc,
            "last_ingested": last_ingested_str,  # Jakarta local, no "+07:00"
        }
        # Include schema only when it exists and we're not on MySQL
        if (schema is not None) and (not omit_schema):
            row["schema"] = schema
        rows.append(row)
    return rows

def _sanitize_filename(name: str) -> str:
    """
    Make a safe filename from database name.
    - Lowercase
    - Replace non-alphanum with underscores
    - Collapse multiple underscores
    """
    if not name:
        return "unknown"
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9\\-_.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Keep one open file handle per database
    writers: Dict[str, Any] = {}
    counts: Dict[str, int] = {}

    def get_writer(db_name: Optional[str]):
        db_key = db_name or "unknown"
        safe = _sanitize_filename(db_key)
        path = os.path.join(OUTPUT_DIR, f"{safe}.jsonl")
        if safe not in writers:
            writers[safe] = open(path, "w", encoding="utf-8")  # overwrites per run
            counts[safe] = 0
        return writers[safe], safe

    try:
        for urn in iter_dataset_urns():
            try:
                rows = fetch_dataset_columns(urn)
                for row in rows:
                    writer, key = get_writer(row.get("database"))
                    writer.write(json.dumps(row, ensure_ascii=False) + "\n")
                    counts[key] += 1
            except Exception as e:
                # If dataset fails, write to a generic 'unknown.jsonl'
                writer, key = get_writer("unknown")
                writer.write(json.dumps({"error": f"{urn}: {e}"}, ensure_ascii=False) + "\n")
                counts[key] += 1
    finally:
        for fh in writers.values():
            try:
                fh.close()
            except Exception:
                pass

    # Summary
    print(f"Exported per database to: {os.path.abspath(OUTPUT_DIR)}")
    for db_file, n in sorted(counts.items()):
        print(f"- {db_file}.jsonl: {n} rows")

if __name__ == "__main__":
    main()

In [9]:
# Smart RAG Ingest: JSONL -> Chroma (Ollama embeddings) with time-aware upsert
# Shows LangGraph inline in a Jupyter notebook (Mermaid only)

import os
import json
import glob
import hashlib
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

from langgraph.graph import StateGraph, END
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

from IPython.display import HTML, display  # for notebook graph rendering

# ----------------- Config (env with SAFE DEFAULTS) -----------------
# Accepts single file, directory, or glob (e.g., './datahub_metadata/*.jsonl')
JSON_INPUT = os.getenv("COLUMNS_OUTPUT_DIR", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR = os.getenv("CHROMA_DIR", "./chroma_db_test")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "test_collection")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
CHAT_MODEL = os.getenv("CHAT_MODEL", "qwen2.5:3b")
TOP_K = int(os.getenv("TOP_K", "5"))

# ----------------- LangGraph state -----------------
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str

# ----------------- Helpers -----------------
def resolve_jsonl_paths(spec: str) -> List[str]:
    """Resolve a single file, directory, or glob into a unique list of .jsonl files."""
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return out

def build_vector_store() -> Chroma:
    embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)
    return Chroma(
        persist_directory=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
    )

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

def _safe_iso_to_dt(s: Optional[str]) -> Optional[datetime]:
    """Parse 'YYYY-MM-DDTHH:MM:SS' safely; returns None if invalid/missing."""
    if not s:
        return None
    try:
        return datetime.fromisoformat(s)
    except Exception:
        return None

def _stable_id(meta: Dict[str, Any], source_name: str) -> str:
    """Stable ID based on database|table|column|source (independent from description)."""
    key = f"{meta.get('database','')}|{meta.get('table','')}|{meta.get('column','')}|{source_name}"
    return hashlib.sha1(key.encode("utf-8")).hexdigest()

def show_graph_mermaid_only(app) -> None:
    """Render LangGraph as a Mermaid diagram inline (no ASCII)."""
    G = app.get_graph()
    mermaid_src = G.draw_mermaid()
    html = f"""
<div class="mermaid">
{mermaid_src}
</div>
<script>
(function () {{
  function renderMermaid() {{
    if (window.mermaid) {{
      mermaid.initialize({{ startOnLoad: true, securityLevel: 'loose' }});
      mermaid.init();
    }}
  }}
  if (!window.mermaid) {{
    var s = document.createElement('script');
    s.src = 'https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js';
    s.onload = renderMermaid;
    document.body.appendChild(s);
  }} else {{
    renderMermaid();
  }}
}})();
</script>
"""
    display(HTML(html))

# ----------------- Nodes -----------------
def ingest_node(state: RAGState) -> RAGState:
    """
    Smart ingest:
      - If collection is empty: embed ALL docs.
      - Else: for each doc, compare incoming meta['last_ingested'] with stored meta['last_ingested'].
              If missing or newer/different -> upsert (delete+add); else skip.
    """
    paths = state["jsonl_paths"]
    vs = build_vector_store()

    total_in_collection = vs._collection.count()
    first_run = (total_in_collection == 0)
    if first_run:
        print("🚀 First run detected (empty collection) — embedding all documents.")

    added = 0
    updated = 0
    skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"⚠️ JSONL not found: {path}")
            continue

        # Build candidate docs + ids
        docs: List[Document] = []
        ids: List[str] = []
        metas: List[Dict[str, Any]] = []

        for raw, obj in read_jsonl(path):
            meta: Dict[str, Any] = obj if isinstance(obj, dict) else {"raw": raw}
            meta["_source"] = p.name
            metas.append(meta)
            ids.append(_stable_id(meta, p.name))
            docs.append(Document(page_content=raw, metadata=meta))

        if not docs:
            continue

        if first_run:
            vs.add_documents(docs, ids=ids)
            added += len(docs)
            print(f"✅ Embedded {len(docs)} docs from {path}")
            continue

        # Subsequent runs: fetch existing metadatas for these ids in chunks
        BATCH = 1000
        existing_meta_by_id: Dict[str, Dict[str, Any]] = {}
        for i in range(0, len(ids), BATCH):
            chunk_ids = ids[i:i + BATCH]
            got = vs._collection.get(ids=chunk_ids, include=["metadatas"])
            got_ids = got.get("ids") or []
            got_metas = got.get("metadatas") or []
            for doc_id, md in zip(got_ids, got_metas):
                existing_meta_by_id[doc_id] = md or {}

        # Decide upsert vs skip
        to_upsert_docs: List[Document] = []
        to_upsert_ids: List[str] = []

        for doc, doc_id, meta in zip(docs, ids, metas):
            incoming_li = meta.get("last_ingested")
            existing = existing_meta_by_id.get(doc_id)

            if not existing:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
                continue

            stored_li = existing.get("last_ingested")
            should_update = (incoming_li != stored_li)

            in_dt = _safe_iso_to_dt(incoming_li)
            st_dt = _safe_iso_to_dt(stored_li)
            if in_dt and st_dt:
                should_update = in_dt > st_dt

            if should_update:
                to_upsert_docs.append(doc)
                to_upsert_ids.append(doc_id)
            else:
                skipped += 1

        if to_upsert_docs:
            vs.delete(ids=to_upsert_ids)
            vs.add_documents(to_upsert_docs, ids=to_upsert_ids)
            updated += len(to_upsert_docs)
            print(f"🔁 Upserted {len(to_upsert_docs)} changed docs from {path}")

    try:
        vs.persist()
    except Exception:
        pass

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    return state

def retrieve_node(state: RAGState) -> RAGState:
    vs = build_vector_store()
    retriever = vs.as_retriever(search_kwargs={"k": TOP_K})
    docs = retriever.invoke(state["question"])  # modern API
    state["context"] = "\n".join(d.page_content for d in (docs or []))
    return state

def generate_node(state: RAGState) -> RAGState:
    llm = ChatOllama(model=CHAT_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.2)
    prompt = f"""You are a helpful assistant.

Use the context (JSON lines) to answer.
If the answer isn't in the context, say so briefly.

Context:
{state.get("context","")}

Question:
{state.get("question","")}

Answer:"""
    resp = llm.invoke(prompt)
    state["answer"] = getattr(resp, "content", None) or str(resp)
    return state

# ----------------- Build graph -----------------
def build_app():
    g = StateGraph(RAGState)
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("generate", generate_node)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

# ----------------- Run once -----------------
def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set COLUMNS_OUTPUT_DIR to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )

    print("📄 JSONL inputs:")
    for p in paths:
        print(" -", p)

    question = input("Enter a test question: ").strip() or "Which columns describe customer identity?"

    app = build_app()
    show_graph_mermaid_only(app)  # 👈 rendered graph only

    init: RAGState = {
        "jsonl_paths": paths,
        "question": question,
        "added": 0,
        "updated": 0,
        "skipped": 0,
        "context": "",
        "answer": "",
    }
    out = app.invoke(init)

    vs = build_vector_store()
    total = vs._collection.count()

    print("\n=== RAG Ingest Report ===")
    print(f"Added (first run): {out.get('added', 0)}")
    print(f"Updated (upserted): {out.get('updated', 0)}")
    print(f"Skipped (unchanged): {out.get('skipped', 0)}")
    print(f"Total docs now in collection: {total}")
    print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")

    print("\n--- Answer ---\n")
    print(out.get("answer", "(no answer)"))
    print("\nDone.")

if __name__ == "__main__":
    main()

📄 JSONL inputs:
 - datahub_metadata/classicmodels.jsonl


Enter a test question:  generate sql query to display 5 rows from customers table



=== RAG Ingest Report ===
Added (first run): 0
Updated (upserted): 0
Skipped (unchanged): 59
Total docs now in collection: 59
Chroma path: /app/chroma_db | Collection: classicmodels

--- Answer ---

Here is an SQL query to display 5 rows from the `customers` table:

```sql
SELECT * FROM customers LIMIT 5;
```

This query will return all columns for the first 5 records in the `customers` table. If you only need specific columns, you can specify them instead of using `*`.

Done.
